In [1]:
import os
import json
import glob
import re

# .jpg ignored because .txt file has the extracted details
DATASET_ROOT = "Datasets/SROIE_Dataset/0325updated.task1train(626p)" # root folder
OUTPUT_FILE = "phase1_sroie_input.jsonl" # json output

In [2]:
def parse_sroie_txt(txt_path):
    """
    Parses SROIE Task 1 format: x1,y1,x2,y2,x3,y3,x4,y4,Transcript
    Returns a text string.
    """
    context_parts = [] # ex - total amount
    
    try:
        with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines() # reads OCR annotations (OCR box)
            
        for line in lines:
            line = line.strip()
            if not line: continue # cleans empty lines
            
            # SROIE lines usually separate coordinates and text by comma
            # But the text itself might contain commas, so we split by the first 8 commas
            parts = line.split(',', 8)
            
            if len(parts) >= 9:
                # Extract coordinates (the location of the text on the receipt)
                coords = parts[:8]
                text = parts[8].strip()
                
                # Convert 8 points to a simple Box [min_x, min_y, max_x, max_y]
                # Coords are strings convert to int
                try:
                    xs = [int(coords[i]) for i in range(0, 8, 2)]
                    ys = [int(coords[i]) for i in range(1, 8, 2)]
                    box = [min(xs), min(ys), max(xs), max(ys)]
                    
                    # Add to context
                    context_parts.append(f"{text} {box}")
                except ValueError:
                    # Fallback if coords are bad so the pipeline wont break
                    context_parts.append(text)
            else:
                # Fallback for lines without coords
                context_parts.append(line)
                
        return " || ".join(context_parts)
        
    except Exception as e:
        print(f"Error reading {txt_path}: {e}")
        return None

In [3]:
print(f"Scanning {DATASET_ROOT} for SROIE files...")

if not os.path.exists(DATASET_ROOT):
    print(f"[!] Error: Path not found.'")
else:
    # Get all .txt files
    txt_files = glob.glob(os.path.join(DATASET_ROOT, "*.txt"))
    print(f"Found {len(txt_files)} receipts.")

    processed_count = 0
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f_out:
        for txt_path in txt_files:
            # Parse
            ocr_context = parse_sroie_txt(txt_path)
            
            if ocr_context:
                entry = {
                    "source": "SROIE",
                    "id": os.path.basename(txt_path).replace('.txt', ''),
                    "input": ocr_context,
                    "output": "TO_BE_GENERATED" 
                }
                f_out.write(json.dumps(entry) + "\n")
                processed_count += 1
                
            if processed_count % 100 == 0:
                print(f"  Processed {processed_count} receipts...")

    print("-" * 30)
    print(f"Done! Saved {processed_count} receipts to {OUTPUT_FILE}")

Scanning Datasets/SROIE_Dataset/0325updated.task1train(626p) for SROIE files...
Found 835 receipts.
  Processed 100 receipts...
  Processed 200 receipts...
  Processed 300 receipts...
  Processed 400 receipts...
  Processed 500 receipts...
  Processed 600 receipts...
  Processed 700 receipts...
  Processed 800 receipts...
------------------------------
Done! Saved 835 receipts to phase1_sroie_input.jsonl
